<a href="https://colab.research.google.com/github/nataliegits/Benchmate/blob/main/aire_card_binder_design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Designing a Binder / Intrabody to the AIRE CARD Domain — Hosted NVIDIA NIM API

Target: **human AIRE** (UniProt **O43918**, 545 aa), **CARD domain, residues 1–95**.

Why CARD: it is an α-helical bundle that drives AIRE self-assembly into nuclear foci/condensates and is
*indispensable* for AIRE function. It's also the most tractable surface for RFdiffusion (helical), and
its AlphaFold model is reliable. Intended use here: an intracellular **research tool / intrabody** that
engages CARD to perturb oligomerization.

Pipeline (all on NVIDIA's **hosted** NIMs — no local GPUs):
1. **Target structure** — pull the ready-made AlphaFold model from EBI and slice out CARD (aa 1–95).
   *(Free + instant; saves AlphaFold2 NIM credits.)*
2. **Hotspot guidance** — compute surface exposure on CARD to help you pick a hotspot patch.
3. **RFdiffusion** — generate binder backbones against CARD.
4. **ProteinMPNN** — design sequences for each backbone.
5. **Boltz-2** — fold binder + CARD complex and score by confidence (AF2-Multimer isn't hosted).


## 1. API key

Get a free key at <https://build.nvidia.com> (open any protein model → **Get API Key**, `nvapi-...`).
One key covers all models.


In [1]:
!pip install requests numpy

In [2]:
import os, time, io, urllib.request
import numpy as np
import requests
from typing import Tuple, Dict, Any, List

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or input("Paste your NVIDIA API key (nvapi-...): ").strip()


Paste your NVIDIA API key (nvapi-...): 

## 2. Hosted endpoint config + async (202) polling

In [3]:
BASE_URL = "https://health.api.nvidia.com/v1"
STATUS_URL = "https://health.api.nvidia.com/v1/status"
ENDPOINTS = {
    "alphafold2":   "protein-structure/alphafold2/predict-structure-from-sequence",
    "rfdiffusion":  "biology/ipd/rfdiffusion/generate",
    "proteinmpnn":  "biology/ipd/proteinmpnn/predict",
    "boltz2":       "biology/mit/boltz2/predict",   # complex prediction + confidence (AF2-multimer not hosted)
}
HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {NVIDIA_API_KEY}",
    "NVCF-POLL-SECONDS": "300",
}

def query_nim(payload, endpoint_key, poll_interval=15, max_wait=3600, echo=False, max_retries=6):
    url = f"{BASE_URL}/{ENDPOINTS[endpoint_key]}"
    if echo: print(f"POST {url}")
    s = requests.Session()
    for attempt in range(max_retries):
        r = s.post(url, headers=HEADERS, json=payload)
        if r.status_code == 200:
            return r.status_code, r.json()
        if r.status_code == 202:
            req_id = r.headers.get("nvcf-reqid")
            print(f"  queued (202), polling {req_id} ...")
            waited = 0
            while waited < max_wait:
                time.sleep(poll_interval); waited += poll_interval
                p = s.get(f"{STATUS_URL}/{req_id}", headers=HEADERS)
                if p.status_code == 200: return p.status_code, p.json()
                if p.status_code in (202, 429) or p.status_code >= 500:
                    continue
                raise Exception(f"Polling error [{p.status_code}]: {p.text}")
            raise TimeoutError(f"max_wait exceeded for {req_id}")
        if r.status_code == 429 or r.status_code >= 500:   # rate-limited / transient -> back off
            wait = min(60, 5 * 2 ** attempt)
            print(f"  {r.status_code}; backing off {wait}s (retry {attempt+1}/{max_retries})")
            time.sleep(wait); continue
        raise Exception(f"Error [{r.status_code}]: {r.text}")
    raise Exception(f"Gave up after {max_retries} retries on {endpoint_key}")


## 3. AIRE definition + design parameters

`CARD_RANGE` is the domain we keep as the target. Full-length sequence is included for reference /
optional full-protein folding, but we only design against CARD.


In [4]:
UNIPROT_ID = "O43918"   # human AIRE

AIRE_FULL = (
    "MATDAALRRLLRLHRTEIAVAVDSAFPLLHALADHDVVPEDKFQETLHLKEKEGCPQAFH"
    "ALLSWLLTQDSTAILDFWRVLFKDYNLERYGRLQPILDSFPKDVDLSQPRKGRKPPAVPK"
    "ALVPPPRLPTKRKASEEARAAAPAALTPRGTASPGSQLKAKPPKKPESSAEQQRLPLGNG"
    "IQTMSASVQRAVAMSSGDVPGARGAVEGILIQQVFESGGSKKCIQVGGEFYTPSKFEDSG"
    "SGKNKARSSSGPKPLVRAKGAQGAAPGGGEARLGQQGSVPAPLALPSDPQLHQKNEDECA"
    "VCRDGGELICCDGCPRAFHLACLSPPLREIPSGTWRCSSCLQATVQEVQPRAEEPRPQEP"
    "PVETPLPPGLRSAGEEVRGPPGEPLAGMDTTLVYKHLPAPPSAAPLPGLDSSALHPLLCV"
    "GPEGQQNLAPGARCGVCGDGTDVLRCTHCAAAFHWRCHFPAGTSRPGTGLRCRSCSGDVT"
    "PAPVEGVLAPSPARLAPGPAKDDTASHEPALHRDDLESLLSEHTFDGILQWAIQSMARPA"
    "APFPS"
)
assert len(AIRE_FULL) == 545

CARD_START, CARD_END = 1, 95             # CARD domain (1-based, inclusive)
card_sequence = AIRE_FULL[CARD_START-1:CARD_END]

# ---- design knobs ----
# contigs: keep CARD (chain A, res 1-95) as target, then build a binder 60-100 aa long.
contigs            = f"A{CARD_START}-{CARD_END}/0 60-100"
# Literature-derived hotspots: the CARD filament interface. Mouse Aire mutagenesis
# (Huoh et al., Nat Commun 2020) showed R16/E18 and D35/D37 abolish filament formation;
# these map by alignment to human R15/E17 and D34/D36. A binder here blocks oligomerization.
hotspot_res        = ["A15", "A17", "A34", "A36"]   # optionally add "A52","A53"
diffusion_steps    = 50          # default quality (was 20 for quick scouting)
num_seq_per_target = 20
sampling_temp      = [0.1]
ca_only            = False
use_soluble_model  = False
backbones_to_make  = 30          # cast a wide net, then filter hard on ipTM
pairs_to_score     = 15          # fold only the top candidates (credits!)
print(f"CARD ({CARD_START}-{CARD_END}), {len(card_sequence)} aa:\n{card_sequence}")


CARD (1-95), 95 aa:
MATDAALRRLLRLHRTEIAVAVDSAFPLLHALADHDVVPEDKFQETLHLKEKEGCPQAFHALLSWLLTQDSTAILDFWRVLFKDYNLERYGRLQP


## 4. Get the target structure (AlphaFold DB → slice CARD)

Downloads the precomputed AlphaFold model for AIRE and keeps only CARD residues, chain A,
renumbered consistently with `contigs`. We also report mean pLDDT so you can confirm CARD is
well-modeled (expect high confidence for a CARD fold).


In [5]:
import json
# Ask the AlphaFold DB API for the current model URL (version-agnostic; avoids _v4/_v5 404s).
api = f"https://alphafold.ebi.ac.uk/api/prediction/{UNIPROT_ID}"
meta = json.loads(urllib.request.urlopen(api, timeout=60).read().decode())
AFDB_URL = meta[0]["pdbUrl"]
print("Downloading", AFDB_URL)
full_pdb_text = urllib.request.urlopen(AFDB_URL, timeout=60).read().decode()

def slice_chain_a(pdb_text, lo, hi):
    """Keep ATOM records for residues lo..hi, force chain A, return PDB string + pLDDT list."""
    out, plddt = [], []
    for line in pdb_text.splitlines():
        if line.startswith("ATOM"):
            resnum = int(line[22:26])
            if lo <= resnum <= hi:
                out.append(line[:21] + "A" + line[22:])
                if line[12:16].strip() == "CA":
                    plddt.append(float(line[60:66]))
    return "\n".join(out) + "\n", plddt

target_pdb, card_plddt = slice_chain_a(full_pdb_text, CARD_START, CARD_END)
print(f"CARD residues kept: {len(card_plddt)}  |  mean pLDDT: {np.mean(card_plddt):.1f}")
print("(>70 = confident, >90 = very high. Low pLDDT regions are flexible/disordered.)")
print(target_pdb[:160])

# --- Alternative: fold from sequence with the AF2 NIM instead (costs credits) ---
# rc, af2 = query_nim({"sequence": card_sequence, "algorithm": "mmseqs2"}, "alphafold2", echo=True)
# target_pdb = af2[0]


CARD residues kept: 95  |  mean pLDDT: 94.0
(>70 = confident, >90 = very high. Low pLDDT regions are flexible/disordered.)
ATOM      1  N   MET A   1       0.407 -14.643   6.829  1.00 47.97           N  
ATOM      2  CA  MET A   1      -0.873 -14.451   6.106  1.00 47.97           C 


## 4b. (Optional) Surface-exposure guide for picking hotspots

Hotspots steer the binder to a specific patch. This ranks CARD residues by solvent exposure
(fewer Cβ neighbors = more exposed) so you can pick a cluster of exposed residues on one face.
Put your chosen ones into `hotspot_res` above (format `["A23","A27",...]`) and re-run step 3.
Leaving `hotspot_res` empty lets RFdiffusion choose — a fine way to start.


In [6]:
def cb_coords(pdb_text):
    res = {}
    for line in pdb_text.splitlines():
        if line.startswith("ATOM"):
            atom = line[12:16].strip()
            if atom in ("CB", "CA"):
                n = int(line[22:26]); aa = line[17:20].strip()
                xyz = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
                # prefer CB; fall back to CA (glycine)
                if n not in res or atom == "CB":
                    res[n] = (aa, xyz)
    return res

res = cb_coords(target_pdb)
nums = sorted(res)
coords = np.array([res[n][1] for n in nums])
exposure = []
for i, n in enumerate(nums):
    d = np.linalg.norm(coords - coords[i], axis=1)
    neighbors = int((d < 10.0).sum()) - 1     # within 10 A
    exposure.append((n, res[n][0], neighbors))
exposure.sort(key=lambda x: x[2])             # fewest neighbors first = most exposed
print("Most solvent-exposed CARD residues (candidates for a hotspot patch):")
for n, aa, nb in exposure[:15]:
    print(f"  A{n}\t{aa}\tneighbors={nb}")


Most solvent-exposed CARD residues (candidates for a hotspot patch):
  A1	MET	neighbors=3
  A68	THR	neighbors=7
  A84	ASP	neighbors=7
  A95	PRO	neighbors=7
  A26	PHE	neighbors=8
  A40	GLU	neighbors=8
  A41	ASP	neighbors=8
  A3	THR	neighbors=9
  A5	ALA	neighbors=9
  A52	LYS	neighbors=9
  A70	ASP	neighbors=9
  A94	GLN	neighbors=9
  A2	ALA	neighbors=10
  A6	ALA	neighbors=10
  A13	LEU	neighbors=10


## 4c. (Optional, recommended) Derive hotspots from the CARD–CARD interface

The biggest lever on binder quality is telling RFdiffusion *where* to bind. Since the goal is to block
AIRE self-assembly, the ideal epitope is CARD's own oligomerization surface. Here we predict the
**CARD homodimer** with Boltz-2 and read the interface residues straight off it, then use the most-
contacting ones as `hotspot_res`. Trust this only if the dimer's own ipTM is reasonable (>~0.5); CARD may
assemble as a higher-order filament, so a dimer is an approximation — but a principled starting patch.


In [7]:
def parse_mmcif_atoms(cif):
    lines = cif.splitlines(); cols = []; data_start = None
    for i, l in enumerate(lines):
        if l.strip() == "loop_":
            j = i + 1; blk = []
            while j < len(lines) and lines[j].strip().startswith("_atom_site."):
                blk.append(lines[j].strip()); j += 1
            if blk: cols = blk; data_start = j; break
    idx = {c: k for k, c in enumerate(cols)}
    g = lambda names: next((idx["_atom_site." + n] for n in names if "_atom_site." + n in idx), None)
    ci_ch = g(["auth_asym_id", "label_asym_id"]); ci_sq = g(["auth_seq_id", "label_seq_id"])
    ci_x = idx.get("_atom_site.Cartn_x"); ci_y = idx.get("_atom_site.Cartn_y"); ci_z = idx.get("_atom_site.Cartn_z")
    ci_gp = g(["group_PDB"]); out = []
    for l in lines[data_start:]:
        s = l.strip()
        if not s or s.startswith("_") or s == "loop_" or s.startswith("#"): break
        p = s.split()
        if len(p) < len(cols) or (ci_gp is not None and p[ci_gp] != "ATOM"): continue
        out.append((p[ci_ch], int(p[ci_sq]), float(p[ci_x]), float(p[ci_y]), float(p[ci_z])))
    return out

rc, dimer = query_nim({"polymers": [
        {"id": "A", "molecule_type": "protein", "sequence": card_sequence},
        {"id": "B", "molecule_type": "protein", "sequence": card_sequence}],
        "output_format": "mmcif", "diffusion_samples": 1, "recycling_steps": 3}, "boltz2", echo=True)
print("CARD-dimer ipTM:", dimer.get("protein_iptm_scores"))

from collections import Counter
atoms = parse_mmcif_atoms(dimer["structures"][0]["structure"])
A = [a for a in atoms if a[0] == "A"]; B = np.array([a[2:] for a in atoms if a[0] == "B"])
contacts = Counter()
for ch, seq, x, y, z in A:
    if (np.linalg.norm(B - np.array([x, y, z]), axis=1) < 5.0).any(): contacts[seq] += 1
top = [n for n, _ in contacts.most_common(5)]
print("CARD-CARD interface residues (chain A):", sorted(contacts))
print("Suggested hotspot_res =", [f"A{n}" for n in sorted(top)])
# To use: copy the printed list into hotspot_res in section 3, then re-run from section 5.


POST https://health.api.nvidia.com/v1/biology/mit/boltz2/predict
CARD-dimer ipTM: [0.37198078632354736]
CARD-CARD interface residues (chain A): [1, 2, 3, 6, 7, 9, 10, 14, 17, 18, 21, 35, 72, 75, 76, 78, 79, 81, 82, 83, 86, 87, 90, 92, 93, 95]
Suggested hotspot_res = ['A18', 'A78', 'A79', 'A82', 'A93']


## 5. RFdiffusion — generate binder backbones

In [8]:
# Resumable: re-running this cell tops up to backbones_to_make instead of starting over.
try:
    backbones
except NameError:
    backbones = []

while len(backbones) < backbones_to_make:
    i = len(backbones)
    q = {"input_pdb": target_pdb, "contigs": contigs, "diffusion_steps": diffusion_steps,
         "random_seed": 1000 + i}
    if hotspot_res:
        q["hotspot_res"] = hotspot_res
    print(f"RFdiffusion backbone {i+1}/{backbones_to_make} (seed {1000+i}) ...")
    try:
        rc, resp = query_nim(q, "rfdiffusion", echo=(i == 0))
        backbones.append(resp["output_pdb"])
    except Exception as e:
        print("  failed, will retry in 30s:", e); time.sleep(30); continue
    time.sleep(2)   # gentle pacing to avoid 429 rate limits
print(f"Have {len(backbones)} backbones.")


RFdiffusion backbone 1/30 (seed 1000) ...
POST https://health.api.nvidia.com/v1/biology/ipd/rfdiffusion/generate
RFdiffusion backbone 2/30 (seed 1001) ...
RFdiffusion backbone 3/30 (seed 1002) ...
RFdiffusion backbone 4/30 (seed 1003) ...
RFdiffusion backbone 5/30 (seed 1004) ...
RFdiffusion backbone 6/30 (seed 1005) ...
RFdiffusion backbone 7/30 (seed 1006) ...
RFdiffusion backbone 8/30 (seed 1007) ...
RFdiffusion backbone 9/30 (seed 1008) ...
RFdiffusion backbone 10/30 (seed 1009) ...
RFdiffusion backbone 11/30 (seed 1010) ...
RFdiffusion backbone 12/30 (seed 1011) ...
RFdiffusion backbone 13/30 (seed 1012) ...
RFdiffusion backbone 14/30 (seed 1013) ...
RFdiffusion backbone 15/30 (seed 1014) ...
RFdiffusion backbone 16/30 (seed 1015) ...
RFdiffusion backbone 17/30 (seed 1016) ...
RFdiffusion backbone 18/30 (seed 1017) ...
RFdiffusion backbone 19/30 (seed 1018) ...
RFdiffusion backbone 20/30 (seed 1019) ...
RFdiffusion backbone 21/30 (seed 1020) ...
RFdiffusion backbone 22/30 (seed 10

## 6. ProteinMPNN — design sequences for each backbone

In [9]:
all_binders = []
for i, bb in enumerate(backbones):
    q = {"input_pdb": bb, "input_pdb_chains": ["A"], "ca_only": ca_only,
         "use_soluble_model": use_soluble_model, "num_seq_per_target": num_seq_per_target,
         "sampling_temp": sampling_temp}
    rc, resp = query_nim(q, "proteinmpnn", echo=(i == 0))
    seqs = [x.strip() for x in resp["mfasta"].split("\n") if ">" not in x][2:]
    all_binders.extend(seqs)
# de-dup and drop empties
all_binders = [s for s in dict.fromkeys(all_binders) if s]
print(f"Designed {len(all_binders)} unique candidate binder sequences.")
for i, s in enumerate(all_binders[:5]):
    print(f"  {i}: {s}")


POST https://health.api.nvidia.com/v1/biology/ipd/proteinmpnn/predict
Designed 570 unique candidate binder sequences.
  0: SYAAYAAEEAAARERRAAAAAAACAAAAAAAAAATEAAAGVDAAARAATAAAAAAAAAARKAAVEAEFDAAAQAAIARLLAAEAAAA
  1: SYEEYRRREEEERERRAAATAAAAAAAAAAAAAAAAARAGVGAAALAAAAAAAAAAAARATAAEEAAFDARATAKIAALLAAEKAAE
  2: SFAARAAAEAAARAEAAAATAAACAALAAAARAAALAREGVDAAMLRATEEAAAAAAAELTAALEAEFDAAAVAKIARLLAEEEARR
  3: SWEAYAAAEAAAAAAEAAATAAAAAAAAAAATAAAAAAEGVDAAAAAATAAAAAAKAAALTAAREAAHRAAAEAAIARWLAAEEAAR
  4: SFEEYLAAEKAAREARAAANAAACKAAAAAAAAAAAAKAGVGEADAAATAAAAAAAAAKKTAALRAEEDARALAKIAALLAAEEARR


## 7. Boltz-2 — fold & score each binder + CARD complex

The hosted AlphaFold2-Multimer endpoint isn't deployed, so we use **Boltz-2** instead: it predicts the
binder+CARD complex structure and returns an overall **confidence score** (0–1, higher = better) we use
to rank candidates. Each chain is a `protein` polymer; chain A = binder, chain B = CARD.

The first call prints the raw response keys so you can see the exact shape. Output is mmCIF
(open in PyMOL/ChimeraX). Defaults run single-sequence (no MSA) for speed — fine for ranking.


In [10]:
pairs = [[b, card_sequence] for b in all_binders]
boltz_results = [None] * len(pairs)

for n, pair in enumerate(pairs):
    if n >= pairs_to_score: break
    binder_seq, target_seq = pair
    payload = {
        "polymers": [
            {"id": "A", "molecule_type": "protein", "sequence": binder_seq},
            {"id": "B", "molecule_type": "protein", "sequence": target_seq},
        ],
        "output_format": "mmcif",
        "diffusion_samples": 1,
        "recycling_steps": 3,
    }
    print(f"Boltz-2 complex {n+1}/{min(pairs_to_score, len(pairs))} ...")
    rc, resp = query_nim(payload, "boltz2", echo=(n == 0))
    if n == 0:
        print("  response keys:", list(resp.keys()))
    boltz_results[n] = resp


Boltz-2 complex 1/15 ...
POST https://health.api.nvidia.com/v1/biology/mit/boltz2/predict
  response keys: ['structures', 'metrics', 'confidence_scores', 'affinities', 'ptm_scores', 'iptm_scores', 'ligand_iptm_scores', 'protein_iptm_scores', 'complex_plddt_scores', 'complex_iplddt_scores', 'complex_pde_scores', 'complex_ipde_scores', 'chains_ptm_scores', 'pair_chains_iptm_scores', 'pae', 'pde']
Boltz-2 complex 2/15 ...
Boltz-2 complex 3/15 ...
  429; backing off 5s (retry 1/6)
Boltz-2 complex 4/15 ...
  429; backing off 5s (retry 1/6)
Boltz-2 complex 5/15 ...
Boltz-2 complex 6/15 ...
  429; backing off 5s (retry 1/6)
  429; backing off 10s (retry 2/6)
Boltz-2 complex 7/15 ...
Boltz-2 complex 8/15 ...
  429; backing off 5s (retry 1/6)
Boltz-2 complex 9/15 ...
  429; backing off 5s (retry 1/6)
  429; backing off 10s (retry 2/6)
Boltz-2 complex 10/15 ...
  429; backing off 5s (retry 1/6)
Boltz-2 complex 11/15 ...
  429; backing off 5s (retry 1/6)
  429; backing off 10s (retry 2/6)
Boltz-2

## 8. Rank by interface confidence (ipTM), save top complexes (.cif)

For binder triage the key metric is **ipTM** (interface predicted TM-score) — it scores confidence in the
binder–target *interface* specifically, unlike the global `confidence_scores` which also reflects monomer
fold quality. We also report **interface PAE** (mean cross-chain PAE, lower Å = better), pTM, and complex
pLDDT. Rule of thumb: ipTM > 0.8 strong, 0.6–0.8 plausible, < 0.6 weak.


In [11]:
def first(x): return x[0] if isinstance(x, list) and x else x

rows = []
for idx, r in enumerate(boltz_results):
    if r is None: continue
    binder = pairs[idx][0]; blen = len(binder)
    iptm   = first(r.get("protein_iptm_scores") or r.get("iptm_scores"))
    ptm    = first(r.get("ptm_scores"))
    cplddt = first(r.get("complex_plddt_scores"))
    conf   = first(r.get("confidence_scores"))
    ipae = None
    pae = r.get("pae")
    if pae:
        M = np.array(pae); M = M[0] if M.ndim == 3 else M
        A, B = slice(0, blen), slice(blen, blen + len(card_sequence))
        if M.shape[0] >= blen + len(card_sequence):
            ipae = float((M[A, B].mean() + M[B, A].mean()) / 2)
    rows.append((idx, binder, iptm, ptm, cplddt, conf, ipae, r["structures"][0]["structure"]))

rows.sort(key=lambda x: (x[2] if x[2] is not None else -1), reverse=True)  # by ipTM

import os
os.makedirs("aire_card_binders", exist_ok=True)
fmt = lambda v: f"{v:.3f}" if isinstance(v, (int, float)) else "  -  "
print(f"{'rank':<5}{'ipTM':>7}{'pTM':>7}{'cpLDDT':>8}{'iPAE':>7}  binder")
for rank, (idx, binder, iptm, ptm, cplddt, conf, ipae, cif) in enumerate(rows):
    open(f"aire_card_binders/rank{rank}_iptm{(iptm or 0):.3f}.cif", "w").write(cif)
    print(f"{rank:<5}{fmt(iptm):>7}{fmt(ptm):>7}{fmt(cplddt):>8}{fmt(ipae):>7}  {binder[:40]}...")


rank    ipTM    pTM  cpLDDT   iPAE  binder
0      0.791  0.713   0.784    -    SFEEYRRREAEEEAARAAATAAAAAAAAAAAAAAAAAREG...
1      0.733  0.695   0.789    -    SFEEYLAAEKAAREARAAANAAACKAAAAAAAAAAAAKAG...
2      0.729  0.664   0.763    -    SFEEYLRREEAEAAERAAATAAAAAAAAAAATAAAAAAAG...
3      0.659  0.674   0.769    -    SAAARAAAEAAAAAEAAAATAAAAAAAAAAAAAAAAAAAG...
4      0.643  0.644   0.800    -    SYEEYRRREEEERERRAAATAAAAAAAAAAAAAAAAARAG...
5      0.626  0.639   0.771    -    SFEEYLEREKEERERRAAAAAAAAKAAAAAARAATAARAG...
6      0.608  0.692   0.793    -    SFEEYRRKEAEERERKAAATAAACEALAAAARAAAAAREG...
7      0.606  0.599   0.851    -    SYAAYAAEEAAARERRAAAAAAACAAAAAAAAAATEAAAG...
8      0.564  0.586   0.830    -    SFAAYAAAEAAERAEKAAATAAACAAAAAAAAAATEALAG...
9      0.533  0.590   0.785    -    SFEAYAAAEAAARAERAAATAAACAAAAAAAAAATAAAAG...
10     0.495  0.602   0.734    -    SWEAYAAAEAAAAAAEAAATAAAAAAAAAAATAAAAAAEG...
11     0.452  0.599   0.791    -    SFEEYRAAEAAERARRAAATAAAAAAAAAAMAAAAAARAG.

---
### How to iterate
- **Pick hotspots from step 4b**, set `hotspot_res`, and re-run — this is the biggest lever on whether
  the binder hits the oligomerization-relevant face of CARD.
- Boltz-2 confidence is a quality/plausibility proxy, **not measured affinity**. Inspect the actual
  binding pose in PyMOL/ChimeraX (open the saved .cif files) before trusting a rank.
- Generate **many more** backbones (raise `backbones_to_make`) and filter hard before the expensive
  multimer step — that's where credits go.
- Reminder: AIRE is **intracellular/nuclear**, so a binder is a tool/intrabody/degrader-bait, not a
  conventional biologic — delivery (e.g. intrabody expression) is the separate hard problem.
